# Practical 8 — Foundational Data Manipulation with Pandas

**Dataset:** the AB Pay transaction ledger (`abpay.db`, SQLite) — the same data the banking app shows on its statement page.

Covers: loading a dataset, understanding its structure, cleaning missing and duplicate values, filtering and sorting, basic statistics and summary functions.

> `abpay.db` is created and seeded the first time the server runs (`npm start`), so run the app once before this notebook.

## 1. Load the dataset

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../abpay.db")
raw = pd.read_sql_query("SELECT * FROM transactions", conn)
conn.close()

raw.head()

,id,account_id,direction,amount,balance_after,counterparty_number,counterparty_name,method,category,description,reference,created_at
0,1,1,out,128000,18452050,None,BigBasket,UPI,Groceries,BigBasket,None,2026-09-01T15:44:06.511Z
1,2,1,out,45000,18580050,None,Uber India,UPI,Transport,Uber India,None,2026-08-31T15:44:06.512Z
2,3,1,in,8500000,18625050,None,Infosys Ltd — Salary,NEFT,Income,Infosys Ltd — Salary,None,2026-08-30T15:44:06.512Z
3,4,1,out,89900,10125050,None,Swiggy,UPI,Dining,Swiggy,None,2026-08-29T15:44:06.513Z
4,5,1,out,245000,10214950,None,BESCOM Electricity,Bill payment,Utilities,BESCOM Electricity,None,2026-08-27T15:44:06.514Z


## 2. Understand the structure

In [2]:
print("Shape (rows, columns):", raw.shape)
print()
raw.info()

Shape (rows, columns): (19, 12)

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   19 non-null     int64 
 1   account_id           19 non-null     int64 
 2   direction            19 non-null     str   
 3   amount               19 non-null     int64 
 4   balance_after        19 non-null     int64 
 5   counterparty_number  0 non-null      object
 6   counterparty_name    19 non-null     str   
 7   method               19 non-null     str   
 8   category             19 non-null     str   
 9   description          19 non-null     str   
 10  reference            0 non-null      object
 11  created_at           19 non-null     str   
dtypes: int64(4), object(2), str(6)
memory usage: 3.2+ KB


In [3]:
# Unique values per column — tells us which columns are categorical.
raw.nunique()

id                     19
account_id              1
direction               2
amount                 11
balance_after          19
counterparty_number     0
counterparty_name      10
method                  4
category                8
description            10
reference               0
created_at             19
dtype: int64

In [4]:
# Numeric summary. Amounts are stored in paise (1 rupee = 100 paise), as banks do to avoid float rounding.
raw[["amount", "balance_after"]].describe()

,amount,balance_after
count,1.900000e+01,1.900000e+01
mean,5.818211e+05,1.275924e+07
std,1.937399e+06,2.690945e+06
min,1.190000e+04,1.012505e+07
25%,1.990000e+04,1.177980e+07
50%,5.990000e+04,1.200935e+07
75%,1.499000e+05,1.247690e+07
max,8.500000e+06,1.862505e+07


## 3. Simulate a messy export

The app's ledger is clean (it is enforced by database constraints), so to practise cleaning we build a realistic messy copy:
a statement downloaded twice overlaps (duplicate rows), and some rows lose their category / description in transit (missing values).

In [5]:
messy = pd.concat([raw, raw.sample(4, random_state=7)], ignore_index=True)   # 4 duplicated rows
messy.loc[[2, 5, 11], "category"] = None                                        # missing categories
messy.loc[[3, 8], "description"] = None                                         # missing descriptions

print("Rows:", len(messy))
print("Exact duplicate rows:", messy.duplicated().sum())
print("Repeated transaction ids:", messy.duplicated(subset="id").sum())
messy.isna().sum()

Rows: 23
Exact duplicate rows: 2
Repeated transaction ids: 4


id                      0
account_id              0
direction               0
amount                  0
balance_after           0
counterparty_number    23
counterparty_name       0
method                  0
category                3
description             2
reference              23
created_at              0
dtype: int64

## 4. Clean missing and duplicate values

In [6]:
df = messy.copy()

# Columns that are empty in every row carry no information.
empty_cols = df.columns[df.isna().all()].tolist()
print("Dropping all-empty columns:", empty_cols)
df = df.drop(columns=empty_cols)

# Duplicates: a transaction id must appear once. Comparing whole rows would miss a
# copy where one side lost a value, so de-duplicate on the primary key instead.
df = df.drop_duplicates(subset="id", keep="first")

# Fill the gaps with sensible defaults.
df["category"] = df["category"].fillna("Other")                         # the app's own default category
df["description"] = df["description"].fillna(df["counterparty_name"])   # description mirrors the payee

print("Rows after cleaning:", len(df))
print("Repeated ids left:", df.duplicated(subset="id").sum())
df.isna().sum()

Dropping all-empty columns: ['counterparty_number', 'reference']
Rows after cleaning: 19
Repeated ids left: 0


id                   0
account_id           0
direction            0
amount               0
balance_after        0
counterparty_name    0
method               0
category             0
description          0
created_at           0
dtype: int64

In [7]:
# Fix types: paise -> rupees, ISO text -> IST datetime, and a signed amount for cash-flow sums.
df["amount_inr"] = df["amount"] / 100
df["balance_inr"] = df["balance_after"] / 100
# Stored in UTC; shown in Indian time and made timezone-naive so plain date strings compare cleanly.
df["created_at"] = pd.to_datetime(df["created_at"]).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None)
df["signed_inr"] = df["amount_inr"].where(df["direction"] == "in", -df["amount_inr"])
df["month"] = df["created_at"].dt.to_period("M")

df[["created_at", "counterparty_name", "category", "direction", "amount_inr", "signed_inr"]].head()

,created_at,counterparty_name,category,direction,amount_inr,signed_inr
0,2026-09-01 21:14:06.511,BigBasket,Groceries,out,1280.0,-1280.0
1,2026-08-31 21:14:06.512,Uber India,Transport,out,450.0,-450.0
2,2026-08-30 21:14:06.512,Infosys Ltd — Salary,Other,in,85000.0,85000.0
3,2026-08-29 21:14:06.513,Swiggy,Dining,out,899.0,-899.0
4,2026-08-27 21:14:06.514,BESCOM Electricity,Utilities,out,2450.0,-2450.0


In [8]:
# The cleaned data should match the original ledger exactly.
assert len(df) == len(raw)
assert df["id"].is_unique
print("Cleaned data matches the original ledger:", len(df), "transactions")

Cleaned data matches the original ledger: 19 transactions


## 5. Filtering

In [9]:
# Boolean mask: spending above ₹1,000
big_spends = df[(df["direction"] == "out") & (df["amount_inr"] > 1000)]
big_spends[["created_at", "counterparty_name", "category", "amount_inr"]]

,created_at,counterparty_name,category,amount_inr
0,2026-09-01 21:14:06.511,BigBasket,Groceries,1280.0
4,2026-08-27 21:14:06.514,BESCOM Electricity,Utilities,2450.0
6,2026-08-22 21:14:06.514,Amazon India,Shopping,12500.0
11,2026-05-25 21:14:06.518,Cult.fit Membership,Other,1499.0
12,2026-06-24 21:14:06.518,Cult.fit Membership,Other,1499.0
13,2026-07-24 21:14:06.518,Cult.fit Membership,Other,1499.0
14,2026-08-23 21:14:06.519,Cult.fit Membership,Other,1499.0


In [10]:
# .isin() — only shopping and dining
df[df["category"].isin(["Shopping", "Dining"])][["counterparty_name", "category", "amount_inr"]]

,counterparty_name,category,amount_inr
3,Swiggy,Dining,899.0
6,Amazon India,Shopping,12500.0
7,Netflix India,Shopping,199.0
8,Netflix India,Shopping,199.0
9,Netflix India,Shopping,249.0
10,Netflix India,Shopping,249.0
15,Spotify India,Shopping,119.0
16,Spotify India,Shopping,119.0
17,Spotify India,Shopping,119.0
18,Spotify India,Shopping,119.0


In [11]:
# .query() — UPI payments in August 2026
df.query("method == 'UPI' and created_at >= '2026-08-01' and created_at < '2026-09-01'")[
    ["created_at", "counterparty_name", "amount_inr"]]

,created_at,counterparty_name,amount_inr
1,2026-08-31 21:14:06.512,Uber India,450.0
3,2026-08-29 21:14:06.513,Swiggy,899.0
5,2026-08-25 21:14:06.514,Jio Recharge,599.0
18,2026-08-28 21:14:06.520,Spotify India,119.0


In [12]:
# String filter — any payee whose name mentions 'India'
df[df["counterparty_name"].str.contains("India", case=False)][["counterparty_name", "amount_inr"]]

,counterparty_name,amount_inr
1,Uber India,450.0
6,Amazon India,12500.0
7,Netflix India,199.0
8,Netflix India,199.0
9,Netflix India,249.0
10,Netflix India,249.0
15,Spotify India,119.0
16,Spotify India,119.0
17,Spotify India,119.0
18,Spotify India,119.0


## 6. Sorting

In [13]:
# Largest transactions first
df.sort_values("amount_inr", ascending=False)[["counterparty_name", "direction", "amount_inr"]].head(5)

,counterparty_name,direction,amount_inr
2,Infosys Ltd — Salary,in,85000.0
6,Amazon India,out,12500.0
4,BESCOM Electricity,out,2450.0
11,Cult.fit Membership,out,1499.0
14,Cult.fit Membership,out,1499.0


In [14]:
# Multi-key sort: by category A→Z, then newest first within each category
df.sort_values(["category", "created_at"], ascending=[True, False])[
    ["category", "created_at", "counterparty_name", "amount_inr"]].head(10)

,category,created_at,counterparty_name,amount_inr
3,Dining,2026-08-29 21:14:06.513,Swiggy,899.0
0,Groceries,2026-09-01 21:14:06.511,BigBasket,1280.0
2,Other,2026-08-30 21:14:06.512,Infosys Ltd — Salary,85000.0
5,Other,2026-08-25 21:14:06.514,Jio Recharge,599.0
14,Other,2026-08-23 21:14:06.519,Cult.fit Membership,1499.0
13,Other,2026-07-24 21:14:06.518,Cult.fit Membership,1499.0
12,Other,2026-06-24 21:14:06.518,Cult.fit Membership,1499.0
11,Other,2026-05-25 21:14:06.518,Cult.fit Membership,1499.0
18,Shopping,2026-08-28 21:14:06.520,Spotify India,119.0
6,Shopping,2026-08-22 21:14:06.514,Amazon India,12500.0


In [15]:
# nlargest / nsmallest shortcuts
df.nsmallest(3, "amount_inr")[["counterparty_name", "amount_inr"]]

,counterparty_name,amount_inr
15,Spotify India,119.0
16,Spotify India,119.0
17,Spotify India,119.0


## 7. Basic statistical analysis

In [16]:
spend = df[df["direction"] == "out"]["amount_inr"]
print(f"Spending transactions : {spend.count()}")
print(f"Total spent           : ₹{spend.sum():,.2f}")
print(f"Mean / median spend   : ₹{spend.mean():,.2f} / ₹{spend.median():,.2f}")
print(f"Std deviation         : ₹{spend.std():,.2f}")
print(f"Smallest / largest    : ₹{spend.min():,.2f} / ₹{spend.max():,.2f}")

Spending transactions : 18
Total spent           : ₹25,546.00
Mean / median spend   : ₹1,419.22 / ₹524.50
Std deviation         : ₹2,851.76
Smallest / largest    : ₹119.00 / ₹12,500.00


In [17]:
# Spending by category: count, total, average, share of spend
by_cat = (df[df["direction"] == "out"]
          .groupby("category")["amount_inr"]
          .agg(transactions="count", total="sum", average="mean")
          .sort_values("total", ascending=False))
by_cat["share_%"] = (by_cat["total"] / by_cat["total"].sum() * 100).round(1)
by_cat.round(2)

,transactions,total,average,share_%
category,,,,
Shopping,9,13872.0,1541.33,54.3
Other,5,6595.0,1319.00,25.8
Utilities,1,2450.0,2450.00,9.6
Groceries,1,1280.0,1280.00,5.0
Dining,1,899.0,899.00,3.5
Transport,1,450.0,450.00,1.8


In [18]:
# How people pay
df["method"].value_counts()

method
Debit card      9
UPI             8
NEFT            1
Bill payment    1
Name: count, dtype: int64

In [19]:
# Monthly cash flow: money in vs money out, and the net
monthly = df.pivot_table(index="month", columns="direction", values="amount_inr", aggfunc="sum", fill_value=0)
monthly["net"] = monthly.get("in", 0) - monthly.get("out", 0)
monthly

direction,in,out,net
month,,,
2026-05,0.0,1817.0,-1817.0
2026-06,0.0,1817.0,-1817.0
2026-07,0.0,1867.0,-1867.0
2026-08,85000.0,18765.0,66235.0
2026-09,0.0,1280.0,-1280.0


In [20]:
# Correlation between transaction size and the balance left afterwards
df[["amount_inr", "balance_inr"]].corr().round(3)

,amount_inr,balance_inr
amount_inr,1.000,0.496
balance_inr,0.496,1.000


## 8. Summary

In [21]:
top = by_cat.index[0]
print(f"Ledger: {len(df)} transactions from {df['created_at'].min():%d %b %Y} to {df['created_at'].max():%d %b %Y}.")
print(f"Cleaning removed {len(messy) - len(df)} duplicate rows, filled "
      f"{messy['category'].isna().sum()} missing categories and {messy['description'].isna().sum()} missing descriptions.")
print(f"Biggest spending category: {top} (₹{by_cat.loc[top, 'total']:,.2f}, {by_cat.loc[top, 'share_%']}% of spend).")
print(f"Most used payment method: {df['method'].mode()[0]}.")

Ledger: 19 transactions from 20 May 2026 to 01 Sep 2026.
Cleaning removed 4 duplicate rows, filled 3 missing categories and 2 missing descriptions.
Biggest spending category: Shopping (₹13,872.00, 54.3% of spend).
Most used payment method: Debit card.
